In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import model_from_json
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 34s 0us/step


In [ ]:
num_train_per_class = 200
num_test_per_class = 50

x_train_small, y_train_small = [], []
x_test_small, y_test_small = [], []

for cls in range(10):
    idx_train = np.where(y_train.flatten() == cls)[0][:num_train_per_class]
    x_train_small.append(x_train[idx_train])
    y_train_small.append(y_train[idx_train])

    idx_test = np.where(y_test.flatten() == cls)[0][:num_test_per_class]
    x_test_small.append(x_test[idx_test])
    y_test_small.append(y_test[idx_test])

x_train_small = np.vstack(x_train_small)
y_train_small = np.vstack(y_train_small)
x_test_small = np.vstack(x_test_small)
y_test_small = np.vstack(y_test_small)

print("Reduced x_train shape:", x_train_small.shape)
print("Reduced x_test shape:", x_test_small.shape)

Reduced x_train shape: (2000, 32, 32, 3)
Reduced x_test shape: (500, 32, 32, 3)


In [ ]:
y_train_small = tf.keras.utils.to_categorical(y_train_small, 10)
y_test_small = tf.keras.utils.to_categorical(y_test_small, 10)


In [ ]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    rescale=1./255
)

test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow(
    x_train_small, y_train_small, batch_size=32
)

test_generator = test_datagen.flow(
    x_test_small, y_test_small, batch_size=32, shuffle=False
)

In [ ]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable = False

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model = models.Sequential([
    layers.Resizing(224, 224),  # Resize on the fly
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ModelCheckpoint("best_model.h5", save_best_only=True)
]


In [ ]:
history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=5,
    callbacks=callbacks
)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 17s/step - accuracy: 0.1066 - loss: 2.4659 

63/63 ━━━━━━━━━━━━━━━━━━━━ 1333s 21s/step - accuracy: 0.1073 - loss: 2.4633 - val_accuracy: 0.2460 - val_loss: 2.0425
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 17s/step - accuracy: 0.2247 - loss: 2.0933 

63/63 ━━━━━━━━━━━━━━━━━━━━ 1322s 21s/step - accuracy: 0.2248 - loss: 2.0932 - val_accuracy: 0.3560 - val_loss: 1.9197
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 17s/step - accuracy: 0.2708 - loss: 2.0016 

63/63 ━━━━━━━━━━━━━━━━━━━━ 1319s 21s/step - accuracy: 0.2707 - loss: 2.0017 - val_accuracy: 0.3920 - val_loss: 1.8303
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 1308s 21s/step - accuracy: 0.2892 - loss: 1.9495 - val_accuracy: 0.3240 - val_loss: 1.8461
Epoch 5/5
 7/63 ━━━━━━━━━━━━━━━━━━━━ 15:32 17s/step - accuracy: 0.3457 - loss: 1.9008

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=3
)

In [ ]:
loss, accuracy = model.evaluate(test_generator)
print("Test Accuracy:", accuracy)


In [ ]:
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = np.argmax(y_test_small, axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
model.save("final_model.h5")

In [ ]:
# =========================
# 1️⃣ Imports
# =========================
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, classification_report

# =========================
# 2️⃣ Load CIFAR-10
# =========================
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

# =========================
# 3️⃣ Reduce Dataset (Faster Training)
# =========================
num_train_per_class = 100
num_test_per_class = 30

x_train_small, y_train_small = [], []
x_test_small, y_test_small = [], []

for cls in range(10):
    idx_train = np.where(y_train.flatten() == cls)[0][:num_train_per_class]
    idx_test = np.where(y_test.flatten() == cls)[0][:num_test_per_class]

    x_train_small.append(x_train[idx_train])
    y_train_small.append(y_train[idx_train])
    x_test_small.append(x_test[idx_test])
    y_test_small.append(y_test[idx_test])

x_train_small = np.vstack(x_train_small)
y_train_small = np.vstack(y_train_small)
x_test_small = np.vstack(x_test_small)
y_test_small = np.vstack(y_test_small)

print("Train shape:", x_train_small.shape)
print("Test shape:", x_test_small.shape)

# =========================
# 4️⃣ Resize Images ONCE (IMPORTANT)
# =========================
x_train_small = tf.image.resize(x_train_small, (224, 224)).numpy()
x_test_small = tf.image.resize(x_test_small, (224, 224)).numpy()

# Normalize
x_train_small = x_train_small / 255.0
x_test_small = x_test_small / 255.0

# =========================
# 5️⃣ Convert Labels
# =========================
y_train_small = tf.keras.utils.to_categorical(y_train_small, 10)
y_test_small = tf.keras.utils.to_categorical(y_test_small, 10)

# =========================
# 6️⃣ Build VGG16 Model
# =========================
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze base model
base_model.trainable = False

# Custom classifier
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),   # Faster than Flatten
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.summary()

# ==========


In [ ]:
# =========================
# 7️⃣ Compile
# =========================
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# =========================
# 8️⃣ Callbacks
# =========================
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True),
    ModelCheckpoint("best_model.h5", save_best_only=True)
]

# =========================
# 9️⃣ Train (NO generator = faster)
# =========================
history = model.fit(
    x_train_small, y_train_small,
    validation_data=(x_test_small, y_test_small),
    epochs=5,
    batch_size=64,
    callbacks=callbacks
)

# =========================
# 🔟 Evaluate
# =========================
loss, accuracy = model.evaluate(x_test_small, y_test_small)
print("Test Accuracy:", accuracy)

# =========================
# 1️⃣1️⃣ Predictions
# =========================
predictions = model.predict(x_test_small)
y_pred = np.argmax(predictions, axis=1)
y_true = np.argmax(y_test_small, axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))

# =========================
# 1️⃣2️⃣ Save Model
# =========================
model.save("final_model.h5")

print("Model saved successfully!")